# 📊 教育データで EDA ＋ 機械学習 入門

**所要時間：約15分** ／ 対象：手を動かして「予測」を体験したい人

### 今日の問い
> 「**高校の成績や SAT から、大学1年の成績（GPA）を予測できる？**」

使うデータ **satgpa**（学生 1000 人分・列は 6 つだけ）で、
**眺める → 可視化する → 予測する → 評価する** の一周を体験します。

### 流れ
1. データを読み込む
2. EDA：まず「眺める」（要約・分布・相関・散布図）
3. 機械学習：回帰モデルで GPA を予測
4. 評価：どれくらい当たったか／予測の限界

> 🧭 説明を読んだら、各セルの ▶ を上から順に押すだけ。

## 0. 準備（ライブラリと日本語フォント）

In [ ]:
!pip install -q japanize-matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
print('準備OK ✅')

## 1. データを読み込む

**satgpa** データの列の意味：

| 列 | 意味 |
|---|---|
| `sex` | 性別（1 / 2） |
| `sat_v` / `sat_m` | SAT の 言語 / 数学（スコア÷10 の値）|
| `sat_sum` | SAT 合計（= sat_v + sat_m）|
| `hs_gpa` | 高校の GPA（成績、最大 4.0 前後）|
| `fy_gpa` | **大学1年の GPA（これを予測したい）** |

In [ ]:
url = 'https://vincentarelbundock.github.io/Rdatasets/csv/openintro/satgpa.csv'
df = pd.read_csv(url)
df = df.drop(columns=[c for c in df.columns if c.lower() == 'rownames'])
print('形：', df.shape, '（行＝学生数, 列＝項目数）')
df.head()

## 2. EDA：まず「眺める」

分析や予測に飛びつく前に、**データの全体像を必ず目で見ます**。
`describe()` で、各列の 件数・平均・ばらつき（std）・最小〜最大 が分かります。

In [ ]:
df.describe().round(2)

**読み方の例**：`fy_gpa`（大学GPA）は平均 2.47・最小 0・最大 4。`hs_gpa`（高校GPA）は平均 3.2。
こうして「だいたいの範囲」を掴んでから、関係を調べます。

### 分布を見る（ヒストグラム）
それぞれの値が「どのあたりに多いか」を見ます。

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for a, col, t in zip(ax, ['hs_gpa', 'sat_sum', 'fy_gpa'],
                     ['高校GPA', 'SAT合計', '大学1年GPA']):
    a.hist(df[col], bins=20, color='#C8611C', alpha=0.85)
    a.set_title(t); a.set_xlabel(col); a.set_ylabel('人数')
plt.tight_layout(); plt.show()

### 関係を見る（相関と散布図）

**相関**＝2つの値が一緒に動く強さ（+1〜−1）。`fy_gpa`（大学GPA）と各列の相関を見ます。

In [ ]:
corr = df.corr(numeric_only=True)['fy_gpa'].drop('fy_gpa').sort_values(ascending=False)
print('大学GPA（fy_gpa）との相関:')
print(corr.round(3).to_string())

In [ ]:
# 高校GPA と 大学GPA の散布図（一番効いていそうな関係）
plt.figure(figsize=(6, 4.2))
plt.scatter(df['hs_gpa'], df['fy_gpa'], s=16, alpha=0.5, color='#1A6BB0')
plt.title('高校GPA と 大学1年GPA の関係')
plt.xlabel('高校GPA (hs_gpa)')
plt.ylabel('大学1年GPA (fy_gpa)')
plt.grid(alpha=0.3)
plt.show()

**観察**：右上がり ——「高校の成績が良い人ほど、大学の成績も高め」という傾向が見えます。
ただし点はかなり散らばっていて、**例外も多い**ことに注目。

## 3. 機械学習：GPA を予測してみる

高校GPA と SAT合計 から、大学1年GPA を予測する **回帰モデル** を作ります。

**大事な作法**：データを **訓練用（学習に使う）** と **テスト用（採点に使う）** に分けます。
見たことのないテスト用データで採点することで、**本当の予測力**が測れます。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X = df[['hs_gpa', 'sat_sum']]   # 手がかり（説明変数）
y = df['fy_gpa']                # 当てたいもの（目的変数）

# 7割を訓練、3割をテストに分ける
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)     # 訓練データで学習
print('学習おわり ✅')
print('式： 大学GPA ≒ {:.3f}×高校GPA + {:.4f}×SAT合計 + {:.2f}'.format(
      model.coef_[0], model.coef_[1], model.intercept_))

## 4. 評価：どれくらい当たった？

テスト用データ（モデルが見ていないデータ）で採点します。

- **MAE**（平均絶対誤差）：予測が平均で何ポイントずれたか（小さいほど良い）
- **R²**（決定係数）：ばらつきのうち何割を説明できたか（1 に近いほど良い・0 は説明力なし）

In [ ]:
pred = model.predict(X_test)
print(f'MAE（平均のズレ）: {mean_absolute_error(y_test, pred):.3f} ポイント')
print(f'R²（説明できた割合）: {r2_score(y_test, pred):.3f}')

# 実際の値 vs 予測値
plt.figure(figsize=(5.2, 5.2))
plt.scatter(y_test, pred, s=18, alpha=0.5, color='#0F766E')
lim = [0, 4.2]
plt.plot(lim, lim, '--', color='#C0392B', label='完全に当たる線')
plt.xlim(lim); plt.ylim(lim)
plt.xlabel('実際の 大学GPA')
plt.ylabel('予測した 大学GPA')
plt.title('予測 vs 実際')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 5. 解釈：予測の「限界」も体感する

R² はだいたい **0.3〜0.4** くらいになります。これは——

- 高校成績や SAT は、大学GPA を **ある程度は** 説明できる（手がかりにはなる）
- でも **半分以上は説明できない**。点は対角線から大きく外れる人も多い

➡️ 「成績だけで将来は決まらない」。やる気・環境・科目選択など、**データに無い要因**がたくさんあるからです。
**「当たるけど、当たりすぎない」** ——これが機械学習の現実的な感覚です。

> 🧠 数字（R²）を鵜呑みにせず、**散布図で“どれくらい外れるか”を自分の目で見る** ことが大切。

## 6. やってみよう（発展）

- 手がかりを増やす：`X = df[['hs_gpa', 'sat_v', 'sat_m']]` に変えて R² は上がる？
- 別のデータでも試す：下のセルは **学校データ CASchools** で「支出 vs 学力」を見ます。
  「**お金をかけるほど点が上がる**」のか、確かめてみましょう。

In [ ]:
# 発展：学校データ CASchools で『支出』と『貧困率』どちらが学力に効く？
ca = pd.read_csv('https://vincentarelbundock.github.io/Rdatasets/csv/AER/CASchools.csv')
ca['score'] = (ca['read'] + ca['math']) / 2

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(ca['expenditure'], ca['score'], s=16, alpha=0.6, color='#C8611C')
ax[0].set_title(f"支出 と 学力 (相関 {ca['expenditure'].corr(ca['score']):.2f})")
ax[0].set_xlabel('生徒1人あたり支出 ($)'); ax[0].set_ylabel('テスト平均点')
ax[1].scatter(ca['lunch'], ca['score'], s=16, alpha=0.6, color='#1A6BB0')
ax[1].set_title(f"給食補助率(貧困の目安) と 学力 (相関 {ca['lunch'].corr(ca['score']):.2f})")
ax[1].set_xlabel('給食補助の対象生徒の割合 (%)'); ax[1].set_ylabel('テスト平均点')
plt.tight_layout(); plt.show()
print('→ 支出より、家庭の経済状況(給食補助率)の方が、学力と強く関係していることが多い')

---
### まとめ
- **EDA** ＝ まずデータを「眺めて・問いを立てる」（要約・分布・相関・散布図）
- **機械学習** ＝ 訓練データで学び、テストデータで採点して **本当の予測力** を測る
- 予測は万能ではない。**当たる範囲と限界の両方**を、自分の目で確かめる

### 出典・データ
- satgpa（OpenIntro Statistics 由来）／ CASchools（R の AER パッケージ由来）
- Rdatasets: https://vincentarelbundock.github.io/Rdatasets/